In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Configuration
TICKERS = ["TSLA", "COIN", "NKE", "SMCI", "XPEV", "NIO", "GOOGL", "AAPL"]
YEARS_OF_DATA = 2
PROFIT_TARGET = 0.0375
STOP_LOSS = 0.0375
FORWARD_DAYS = 14

In [3]:
# Shared model components
FEATURES = ['RSI', 'SMA1', 'SMA2', 'SMA3', 'MACD', 'Signal_Line',
            'Upper_Band', 'Lower_Band', 'Volume_MA20',
            '5_day_return', '10_day_return', 'Volatility']

results = []

def get_stock_data(ticker, start_date, end_date):
    print("Getting data for:   ", ticker)
    df = yf.download(ticker, start=start_date, end=end_date + timedelta(days=1), 
                     interval='1d', auto_adjust=False, progress=False)
    df = df.reset_index()
    df['Date'] = pd.to_datetime(df['Date'])
    df.set_index('Date', inplace=True)
    df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
    return df

def add_technical_indicators(df):
    df['SMA1'] = df['Close'].rolling(window=20).mean()
    df['SMA2'] = df['Close'].rolling(window=50).mean()
    df['SMA3'] = df['Close'].rolling(window=200).mean()
    delta = df['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(window=14).mean()
    loss = -delta.where(delta < 0, 0).rolling(window=14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    ema12 = df['Close'].ewm(span=12, adjust=False).mean()
    ema26 = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = ema12 - ema26
    df['Signal_Line'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['Upper_Band'] = df['SMA1'] + (2 * df['Close'].rolling(20).std())
    df['Lower_Band'] = df['SMA1'] - (2 * df['Close'].rolling(20).std())
    df['Volume_MA20'] = df['Volume'].rolling(window=20).mean()
    return df

def compute_expected_return(df):
    df['5_day_return'] = df['Close'].pct_change(5)
    df['10_day_return'] = df['Close'].pct_change(10)
    df['Volatility'] = df['Close'].rolling(5).std()
    df['Expected_Return'] = np.nan
    close_prices = df['Close'].values
    for i in range(len(close_prices) - FORWARD_DAYS):
        current_price = close_prices[i]
        future_max = np.nanmax(close_prices[i + 1:i + 1 + FORWARD_DAYS])
        expected_return = (future_max - current_price) / current_price
        df.iloc[i, df.columns.get_loc('Expected_Return')] = expected_return
    return df

end_date = datetime.now()
start_date = end_date - timedelta(days=365 * YEARS_OF_DATA)

for ticker in TICKERS:
    try:
        df = get_stock_data(ticker, start_date, end_date)
        df['Volume'] = pd.to_numeric(df['Volume'], errors='coerce')
        df = add_technical_indicators(df)
        df = compute_expected_return(df)
        df_model = df.dropna(subset=FEATURES + ['Expected_Return'])
        if len(df_model) < 50:
            continue

        X = df_model[FEATURES]
        y = df_model['Expected_Return']
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        model = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)
        model.fit(X_scaled, y)

        latest = df.iloc[[-1]]
        if latest[FEATURES].isnull().values.any():
            continue
        latest_scaled = scaler.transform(latest[FEATURES])
        predicted_return = model.predict(latest_scaled)[0]

        entry_price = latest['Close'].values[0]
        tp = entry_price * (1 + PROFIT_TARGET)
        sl = entry_price * (1 - STOP_LOSS)

        results.append({
            "Ticker": ticker,
            "Date": latest.index[-1].date(),
            "Price": round(entry_price, 2),
            "Max (%)": round(predicted_return * 100, 2),
            "Max (TP)": round(entry_price + entry_price*predicted_return, 2),
            "TP": round(tp, 2),
            "SL": round(sl, 2),
            "Entry Signal": "✅ Entry" if predicted_return >= PROFIT_TARGET else "❌ No Entry"
        })

    except Exception as e:
        print(f"Error processing {ticker}: {e}")

'''
# Display results
results_df = pd.DataFrame(results)
print("\n=== Multi-Ticker Prediction Table ===")
print(results_df.to_string(index=False))
'''

from tabulate import tabulate
results_df = pd.DataFrame(results)
print("\n=== Multi-Ticker Prediction Table ===")
print(tabulate(results_df, headers='keys', tablefmt='plain'))  # or 'grid', 'tsv', 'simple'


Getting data for:    TSLA
Getting data for:    COIN
Getting data for:    NKE
Getting data for:    SMCI
Getting data for:    XPEV
Getting data for:    NIO
Getting data for:    GOOGL
Getting data for:    AAPL

=== Multi-Ticker Prediction Table ===
    Ticker    Date          Price    Max (%)    Max (TP)      TP      SL  Entry Signal
 0  TSLA      2025-06-11   330.05      18.73      391.88  342.43  317.68  ✅ Entry
 1  COIN      2025-06-11   255.62       2.93      263.1   265.21  246.03  ❌ No Entry
 2  NKE       2025-06-11    63.21       7.98       68.25   65.58   60.83  ✅ Entry
 3  SMCI      2025-06-11    43.44       3.07       44.77   45.07   41.81  ❌ No Entry
 4  XPEV      2025-06-11    20.53       8.46       22.27   21.31   19.76  ✅ Entry
 5  NIO       2025-06-11     3.81       6.71        4.06    3.95    3.66  ✅ Entry
 6  GOOGL     2025-06-11   177.93       4.27      185.52  184.6   171.26  ✅ Entry
 7  AAPL      2025-06-11   200.75       3.2       207.18  208.28  193.22  ❌ No Entry


### Improvements

Handles randomforests well - check prediction after

In [4]:
# Core features
BASE_FEATURES = ['RSI', 'SMA1', 'SMA2', 'SMA3', 'MACD', 'Signal_Line',
                 'Upper_Band', 'Lower_Band', 'Volume_MA20',
                 '5_day_return', '10_day_return', 'Volatility']

results = []

def add_lagged_features(df, lags=[1, 2, 3]):
    for lag in lags:
        for feature in BASE_FEATURES:
            if feature in df.columns:
                df[f"{feature}_lag{lag}"] = df[feature].shift(lag)
    return df

# Time window
end_date = datetime.now()
start_date = end_date - timedelta(days=365 * YEARS_OF_DATA)

for ticker in TICKERS:
    try:
        df = get_stock_data(ticker, start_date, end_date)
        df['Volume'] = pd.to_numeric(df['Volume'], errors='coerce')
        df = add_technical_indicators(df)
        df = compute_expected_return(df)
        df = add_lagged_features(df)

        df.fillna(method='ffill', inplace=True)
        df.dropna(inplace=True)

        all_features = BASE_FEATURES + [f"{feat}_lag{lag}" for feat in BASE_FEATURES for lag in [1, 2, 3]]
        df_model = df.dropna(subset=all_features + ['Expected_Return'])

        if len(df_model) < 50:
            continue

        # Optional: use recent data only
        df_model = df_model[-180:]

        X = df_model[all_features]
        y = df_model['Expected_Return']
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # Time-aware train/test split
        train_size = int(0.8 * len(X_scaled))
        X_train, X_test = X_scaled[:train_size], X_scaled[train_size:]
        y_train, y_test = y[:train_size], y[train_size:]

        model = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))
        #print(f"{ticker} - RMSE: {rmse:.4f}")

        latest = df.iloc[[-1]]
        if latest[all_features].isnull().values.any():
            continue

        latest_scaled = scaler.transform(latest[all_features])
        predicted_return = model.predict(latest_scaled)[0]

        # Confidence (std dev) from trees
        predictions = [tree.predict(latest_scaled)[0] for tree in model.estimators_]
        std_dev = np.std(predictions)

        entry_price = latest['Close'].values[0]
        tp = entry_price * (1 + PROFIT_TARGET)
        sl = entry_price * (1 - STOP_LOSS)

        results.append({
            "Ticker": ticker,
            "Date": latest.index[-1].date(),
            "Price": round(entry_price, 2),
            "Max (%)": round(predicted_return * 100, 2),
            "Max (TP)": round(entry_price + entry_price*predicted_return, 2),
            "TP": round(tp, 2),
            "SL": round(sl, 2),
            "Std Dev": round(std_dev, 4),
            "Entry Signal": "✅ Entry" if predicted_return >= PROFIT_TARGET else "❌ No Entry"
        })

        # Optional: display top features
        importances = model.feature_importances_
        important_features = sorted(zip(X.columns, importances), key=lambda x: x[1], reverse=True)
        '''
        print(f"{ticker} - Top Features:")
        for f, score in important_features[:5]:
            print(f"  {f}: {score:.4f}")
        '''

    except Exception as e:
        print(f"Error processing {ticker}: {e}")

# Display results
df_results = pd.DataFrame(results)
print("\n=== Multi-Ticker Prediction Table (Modified) ===")
print(tabulate(df_results, headers='keys', tablefmt='plain'))

Getting data for:    TSLA
Getting data for:    COIN
Getting data for:    NKE
Getting data for:    SMCI
Getting data for:    XPEV
Getting data for:    NIO
Getting data for:    GOOGL
Getting data for:    AAPL

=== Multi-Ticker Prediction Table (Modified) ===
    Ticker    Date          Price    Max (%)    Max (TP)      TP      SL    Std Dev  Entry Signal
 0  TSLA      2025-06-11   330.1       20.64      398.24  342.48  317.72     0.0635  ✅ Entry
 1  COIN      2025-06-11   255.6        5.75      270.3   265.18  246.01     0.0562  ✅ Entry
 2  NKE       2025-06-11    63.2       10.97       70.13   65.57   60.83     0.0523  ✅ Entry
 3  SMCI      2025-06-11    43.4        5.51       45.79   45.03   41.77     0.0472  ✅ Entry
 4  XPEV      2025-06-11    20.52      32.86       27.27   21.29   19.76     0.1057  ✅ Entry
 5  NIO       2025-06-11     3.8       20.69        4.58    3.94    3.65     0.0394  ✅ Entry
 6  GOOGL     2025-06-11   177.94       5         186.84  184.61  171.27     0.0488  ✅ 

In [5]:
'''
from sklearn.model_selection import cross_val_score

for n in [10, 100, 200, 300]:
    model = RandomForestRegressor(n_estimators=n, max_depth=5, random_state=42)
    scores = cross_val_score(model, X_scaled, y, cv=3, scoring='neg_mean_squared_error')
    print(f"{n} trees: Avg RMSE = {(-scores.mean())**0.5:.4f}")

seeds = [0, 21, 42, 84, 123]
for s in seeds:
    model = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=s)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))
    print(f"Seed {s} -> RMSE: {rmse:.4f}")
'''

'\nfrom sklearn.model_selection import cross_val_score\n\nfor n in [10, 100, 200, 300]:\n    model = RandomForestRegressor(n_estimators=n, max_depth=5, random_state=42)\n    scores = cross_val_score(model, X_scaled, y, cv=3, scoring=\'neg_mean_squared_error\')\n    print(f"{n} trees: Avg RMSE = {(-scores.mean())**0.5:.4f}")\n\nseeds = [0, 21, 42, 84, 123]\nfor s in seeds:\n    model = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=s)\n    model.fit(X_train, y_train)\n    y_pred = model.predict(X_test)\n    rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))\n    print(f"Seed {s} -> RMSE: {rmse:.4f}")\n'